In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path
import json

from satelliteSegmentation.tokenizer import IDX2LABEL, NICECOLOR
from satelliteSegmentation.utils import plot_confusion_matrix

In [ ]:
def plot_compare(hist: dict[str, pd.DataFrame]):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax = ax.flatten()

    for name, df in hist.items():
        ax[0].plot(range(1, len(df) + 1), df["val_loss"], label=name)

    mins = pd.DataFrame(
        [[name, min(hist[name].val_loss)] for name in hist.keys()],
        columns=["model", "val_loss"],
    )

    sns.barplot(
        mins,
        y="model",
        x="val_loss",
        orient="h",
        order=mins.sort_values("val_loss")["model"],
        ax=ax[1],
    )

    # Add value to each bar
    for bar in ax[1].patches:
        value = bar.get_width()
        ax[1].text(
            value,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.4f}",
            va="center",
            ha="right",
            fontsize=10,
        )

    ax[0].legend()
    plt.tight_layout()
    plt.show()


def open_hist(folder: Path | str):
    folder = folder if isinstance(folder, Path) else Path(folder)

    hist: dict[str, pd.DataFrame] = {}
    for model_folder in folder.glob("*"):
        if model_folder.is_file():
            raise RuntimeError(
                f"Found a file while searching for model folders. Folder at: {model_folder.resolve().as_posix()}"
            )

        if not model_folder.name.startswith("_"):
            hist[model_folder.name] = pd.read_csv(model_folder / "hist.csv")

    return hist

## Comparar Modelos

In [ ]:
hist_lr = open_hist("../results/lr")

In [ ]:
plot_compare(hist_lr)

In [ ]:
ax = sns.pointplot(
    pd.DataFrame([[float(k.replace("lr-", "").replace("baseline", "1e-4")) , v["val_loss"].min()] for k, v in hist_lr.items()], columns=["lr", "val_loss"]),
    x="lr", y="val_loss",
);
plt.xticks(rotation=45);

## Kernel

In [ ]:
hist_kernel = open_hist("../results/k")

In [ ]:
plot_compare(hist_kernel)

## Weight Decay

In [ ]:
hist_wd = open_hist("../results/wd")

In [ ]:
plot_compare(hist_wd)

## Cross Validation

In [ ]:
with open(Path("../results/unet/cross-val/models.json"), "r") as f:
    cross_val: dict = json.load(f)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax = ax.flatten()

dice_scores = []
for fold in cross_val.keys():
    
    dice_scores.append(cross_val[fold]["metrics"]["dice"])
    ax[0].barh(fold, cross_val[fold]["metrics"]["dice"])
print(f"Dice Std: {np.std(dice_scores):.2f}")
print(f"Dice Mean: {np.mean(dice_scores):.2f}")
ax[0].axvline(np.mean(dice_scores), c="black", linestyle="--", label=f"mean {np.mean(dice_scores):.2f}")
ax[0].legend()
ax[0].set_title("Dice score")

mean_iou = []
for fold in cross_val.keys():
    mean_iou.append(cross_val[fold]["metrics"]["miou"])
    ax[1].barh(fold, cross_val[fold]["metrics"]["miou"])
print(f"IoU Std: {np.std(mean_iou):.2f}")
print(f"IoU Mean: {np.mean(mean_iou):.2f}")
ax[1].axvline(np.mean(mean_iou), c="black", linestyle="--", label=f"mean {np.mean(mean_iou):.2f}")
ax[1].legend()
ax[1].set_title("IoU score")

pixel_acc_scores = []
for fold in cross_val.keys():
    pixel_acc_scores.append(cross_val[fold]["metrics"]["pixel_acc"])
    ax[2].barh(fold, cross_val[fold]["metrics"]["pixel_acc"])
print(f"Pixel Acc Std: {np.std(pixel_acc_scores):.2f}")
print(f"Pixel Acc Mean: {np.mean(pixel_acc_scores):.2f}")
ax[2].axvline(np.mean(pixel_acc_scores), c="black", linestyle="--", label=f"mean {np.mean(pixel_acc_scores):.2f}")
ax[2].legend()
ax[2].set_title("Pixel Accuracy");

In [ ]:
mean_dice_per_class = np.zeros(6)
for fold in cross_val.keys():
    for cls_idx, cls_val in enumerate(
        filter(
            lambda a: not pd.isna(a),
            cross_val[fold]["metrics"]["dice_per_class"],
        )
    ):
        mean_dice_per_class[cls_idx] += cls_val
mean_dice_per_class /= 5

In [ ]:
mean_iou_per_class = np.zeros(6)
for fold in cross_val.keys():
    for cls_idx, cls_val in enumerate(
        filter(
            lambda a: not pd.isna(a),
            cross_val[fold]["metrics"]["iou_per_class"],
        )
    ):
        mean_iou_per_class[cls_idx] += cls_val
mean_iou_per_class /= 5

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax = ax.flatten()

for cls_idx, cls_val in enumerate(mean_dice_per_class):
    ax[0].barh(IDX2LABEL[cls_idx], cls_val, color=NICECOLOR[cls_idx])
ax[0].set_title("Mean Dice per Class")

for cls_idx, cls_val in enumerate(mean_iou_per_class):
    ax[1].barh(IDX2LABEL[cls_idx], cls_val, color=NICECOLOR[cls_idx])
ax[1].set_title("Mean IoU per Class")

;

In [ ]:
cross_val.keys()

In [ ]:
mean_cm = np.zeros((7, 7))
for f in cross_val.keys():
    mean_cm += np.array(cross_val[f]["metrics"]["confusion_matrix"])
mean_cm

In [ ]:
plot_confusion_matrix(mean_cm)